In [1]:
!pip install genesis-world==0.3.10

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [2]:
"""
Phase-1 Working code with fixed errors.
"""

import genesis as gs
import numpy as np
import math
import os
import imageio.v2 as imageio

# ===================== 0. RESET =====================
try:
    gs.destroy()
except Exception:
    pass

# ===================== 1. INIT =====================
gs.init(backend=gs.cuda, logging_level="warning")

# ===================== 2. OPTIONS (FAST) =====================
sim_dt = 1.0 / 30.0
video_fps = 10
RES_VIS = (320, 240)
steps_per_frame = max(1, int(round((1.0 / video_fps) / sim_dt)))

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=sim_dt, substeps=1),
    show_viewer=False,
    renderer=gs.renderers.Rasterizer(),
)
scene.add_entity(gs.morphs.Plane())

# ===================== 3. OVAL TRACK =====================
OVAL_RADIUS = 15.0
STRAIGHT_LEN = 40.0
TRACK_WIDTH = 10.0
NUM_CONES = 60
perimeter = 2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS

CONE_R = 0.30
LANE_HALF = TRACK_WIDTH / 2.0
CAR_HALF_W = 1.25
LANE_SAFE = float(LANE_HALF - (CAR_HALF_W + CONE_R + 0.25))
print("LANE_SAFE =", LANE_SAFE)

def wrap_pi(a):
    return (a + math.pi) % (2 * math.pi) - math.pi

def get_oval_pos(dist, offset=0.0):
    d = dist % perimeter
    if d < STRAIGHT_LEN:
        return (OVAL_RADIUS + offset, -STRAIGHT_LEN / 2 + d)
    d -= STRAIGHT_LEN
    if d < math.pi * OVAL_RADIUS:
        ang = (d / (math.pi * OVAL_RADIUS)) * math.pi
        R = (OVAL_RADIUS + offset)
        return (R * math.cos(ang), STRAIGHT_LEN / 2 + R * math.sin(ang))
    d -= math.pi * OVAL_RADIUS
    if d < STRAIGHT_LEN:
        return (-OVAL_RADIUS - offset, STRAIGHT_LEN / 2 - d)
    d -= STRAIGHT_LEN
    ang = math.pi + (d / (math.pi * OVAL_RADIUS)) * math.pi
    R = (OVAL_RADIUS + offset)
    return (R * math.cos(ang), -STRAIGHT_LEN / 2 + R * math.sin(ang))

def get_tangent_normal(dist, eps=0.25):
    x1, y1 = get_oval_pos(dist, 0.0)
    x2, y2 = get_oval_pos(dist + eps, 0.0)
    tx, ty = (x2 - x1), (y2 - y1)
    nrm = math.sqrt(tx * tx + ty * ty) + 1e-8
    tx, ty = tx / nrm, ty / nrm
    nx, ny = -ty, tx
    return (tx, ty), (nx, ny)

def project_to_centerline(px, py, s_guess, window=16.0, samples=21):
    best_s = s_guess
    best_d2 = 1e30
    s0 = s_guess - window
    ds = (2 * window) / (samples - 1)
    for i in range(samples):
        s = s0 + i * ds
        cx, cy = get_oval_pos(s, 0.0)
        dx, dy = px - cx, py - cy
        d2 = dx * dx + dy * dy
        if d2 < best_d2:
            best_d2 = d2
            best_s = s
    best_s = best_s % perimeter
    cx, cy = get_oval_pos(best_s, 0.0)
    (tx, ty), (nx, ny) = get_tangent_normal(best_s)
    cte = (px - cx) * nx + (py - cy) * ny
    return best_s, cx, cy, tx, ty, nx, ny, float(cte)

# ===================== 3.1 CONES =====================
cone_entities = []
cone_xy = []  # (x,y,r)

for i in range(NUM_CONES):
    dist = (i / NUM_CONES) * perimeter

    ix, iy = get_oval_pos(dist, offset=-LANE_HALF)
    cone_entities.append(scene.add_entity(
        gs.morphs.Cylinder(radius=CONE_R, height=1.0, pos=(ix, iy, 0.5)),
        surface=gs.surfaces.Default(color=(1, 1, 0), emissive=(0.25, 0.25, 0.0)),
        material=gs.materials.Rigid(friction=2.0),
    ))
    cone_xy.append((float(ix), float(iy), float(CONE_R)))

    ox, oy = get_oval_pos(dist, offset=+LANE_HALF)
    cone_entities.append(scene.add_entity(
        gs.morphs.Cylinder(radius=CONE_R, height=1.0, pos=(ox, oy, 0.5)),
        surface=gs.surfaces.Default(color=(1, 1, 0), emissive=(0.25, 0.25, 0.0)),
        material=gs.materials.Rigid(friction=2.0),
    ))
    cone_xy.append((float(ox), float(oy), float(CONE_R)))

cone_xy = np.array(cone_xy, dtype=np.float32)

# ===================== 3.5 OBSTACLES =====================
OB_SIZE = 1.5
obstacle_entities = []
obs_boxes = []  # (cx,cy,hx,hy)

obs_dists = [0.12*perimeter, 0.22*perimeter, 0.34*perimeter, 0.48*perimeter, 0.65*perimeter, 0.85*perimeter]
obs_lat   = [ +1.6,          -1.6,          +2.0,          -2.0,          +1.5,          -1.5 ]

for d, lat in zip(obs_dists, obs_lat):
    (_, _), (nx, ny) = get_tangent_normal(d)
    cx, cy = get_oval_pos(d, 0.0)
    ox = cx + lat * nx
    oy = cy + lat * ny

    obstacle_entities.append(scene.add_entity(
        gs.morphs.Box(size=(OB_SIZE, OB_SIZE, OB_SIZE), pos=(ox, oy, OB_SIZE / 2)),
        surface=gs.surfaces.Default(color=(1, 0, 0), emissive=(0.35, 0.0, 0.0)),
        material=gs.materials.Rigid(friction=1.0),
    ))

    half = OB_SIZE / 2.0
    obs_boxes.append((float(ox), float(oy), float(half), float(half)))

obs_boxes = np.array(obs_boxes, dtype=np.float32)

# ===================== 4. CAR =====================
start_x, start_y = get_oval_pos(0.0, 0.0)

car = scene.add_entity(
    gs.morphs.URDF(file="assets/smart_car.urdf", fixed=True, pos=(start_x, start_y - 5.0, 0.8)),
    material=gs.materials.Rigid(friction=1.5),
)

# ===================== 5. CAMERAS =====================
# Top camera (bird's eye view of entire track)
track_center_x = 0.0
track_center_y = 0.0
track_width_x = 2 * (OVAL_RADIUS + TRACK_WIDTH + 5)
track_width_y = STRAIGHT_LEN + 2 * (OVAL_RADIUS + TRACK_WIDTH + 5)
cam_height = max(track_width_x, track_width_y) * 1.2

cam_top = scene.add_camera(
    res=(640, 480),
    pos=(track_center_x, track_center_y, cam_height),
    lookat=(track_center_x, track_center_y, 0),
    fov=50,
    far=2000
)

# In-car camera (manually positioned relative to car)
# Camera offset from car center (URDF definition: x=0, y=0.9, z=1.05)
CAM_OFFSET_X = 0.0
CAM_OFFSET_Y = 0.9
CAM_OFFSET_Z = 1.05

cam_incar = scene.add_camera(
    res=RES_VIS,
    pos=(start_x, start_y - 5.0, 0.8 + CAM_OFFSET_Z),
    lookat=(start_x, start_y - 4.0, 0.8 + CAM_OFFSET_Z),
    fov=75,
    far=1500
)

scene.build()
scene.step()

# ===================== 6. HELPERS =====================
def to_np(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def yaw_to_quat_wxyz(yaw):
    """
    CORRECTED: Negative yaw for proper steering direction.
    In coordinate system (+Y forward, +X right, +Z up):
    - Positive steering angle = LEFT turn = CCW rotation = NEGATIVE yaw
    - Negative steering angle = RIGHT turn = CW rotation = POSITIVE yaw
    """
    h = 0.5 * float(-yaw)  # SIGN FLIP HERE
    return (math.cos(h), 0.0, 0.0, math.sin(h))

def set_pose_yaw(ent, px, py, pz, yaw):
    """Set entity pose with yaw angle. Consolidated quaternion conversion."""
    ent.set_pos((float(px), float(py), float(pz)))
    ent.set_quat(yaw_to_quat_wxyz(yaw))

def set_joint_positions(vals):
    if not hasattr(car, "set_dofs_position"):
        return
    car.set_dofs_position(
        np.array(vals, dtype=np.float32),
        dofs_idx_local=np.array([0, 1], dtype=np.int32)
    )

def update_incar_camera(cam, car_x, car_y, car_z, car_yaw):
    """
    Update in-car camera position based on car pose.
    Camera is positioned at driver's eye level, looking forward.
    """
    # Rotate camera offset by car's yaw
    cos_yaw = math.cos(car_yaw)
    sin_yaw = math.sin(car_yaw)
    
    # Transform camera offset from car's local frame to world frame
    cam_world_x = car_x + CAM_OFFSET_X * cos_yaw - CAM_OFFSET_Y * sin_yaw
    cam_world_y = car_y + CAM_OFFSET_X * sin_yaw + CAM_OFFSET_Y * cos_yaw
    cam_world_z = car_z + CAM_OFFSET_Z
    
    # Lookat point is ahead of the camera in the direction the car is facing
    lookat_dist = 10.0
    lookat_x = cam_world_x + lookat_dist * sin_yaw
    lookat_y = cam_world_y + lookat_dist * cos_yaw
    lookat_z = cam_world_z
    
    cam.set_pose(
        pos=(cam_world_x, cam_world_y, cam_world_z),
        lookat=(lookat_x, lookat_y, lookat_z)
    )

# Pin static objects by pose (pos/quat) — quat is also (w,x,y,z) here
pinned_pose = []
for ent in cone_entities + obstacle_entities:
    p = to_np(ent.get_pos()).reshape(-1).copy()
    q = to_np(ent.get_quat()).reshape(-1).copy()
    pinned_pose.append((ent, p, q))

# ===================== 7. LIDAR (FAST) =====================
N_RAYS = 41
FOV = math.radians(170.0)
ray_angles = np.linspace(-FOV / 2, FOV / 2, N_RAYS).astype(np.float32)
LIDAR_MAX = 30.0

def lidar_scan_fast(px, py, yaw, ray_angles, obs_boxes, max_range=30.0):
    if len(obs_boxes) == 0:
        return np.full(len(ray_angles), max_range, dtype=np.float32)

    ga = ray_angles + yaw
    dx = np.sin(ga).astype(np.float32)
    dy = np.cos(ga).astype(np.float32)

    P = np.array([px, py], dtype=np.float32).reshape(1, 1, 2)
    D = np.stack([dx, dy], axis=1).reshape(-1, 1, 2)

    B = obs_boxes.reshape(1, -1, 4)
    Bmin = B[..., :2] - B[..., 2:]
    Bmax = B[..., :2] + B[..., 2:]

    invD = 1.0 / (D + 1e-9)
    t0 = (Bmin - P) * invD
    t1 = (Bmax - P) * invD

    t_enter = np.max(np.minimum(t0, t1), axis=2)
    t_exit  = np.min(np.maximum(t0, t1), axis=2)

    hit = (t_exit > t_enter) & (t_exit > 0)
    dists = np.where(hit, np.maximum(0.0, t_enter), np.inf)
    return np.clip(np.min(dists, axis=1), 0.0, max_range).astype(np.float32)

# ===================== 8. CLEARANCE =====================
def dist_point_to_aabb(px, py, cx, cy, hx, hy):
    dx = max(abs(px - cx) - hx, 0.0)
    dy = max(abs(py - cy) - hy, 0.0)
    return math.hypot(dx, dy)

def clearance_at_point(px, py):
    best = 1e9
    for (cx, cy, hx, hy) in obs_boxes:
        best = min(best, dist_point_to_aabb(px, py, float(cx), float(cy), float(hx), float(hy)))
    dx = cone_xy[:, 0] - px
    dy = cone_xy[:, 1] - py
    center_dist = np.sqrt(dx*dx + dy*dy)
    best = min(best, float(np.min(center_dist - cone_xy[:, 2])))
    return float(best)

# ===================== 9. BICYCLE MODEL =====================
WHEELBASE = 2.8

def integrate_bicycle(px, py, yaw, v, delta, dt):
    yaw_rate = (v / WHEELBASE) * math.tan(delta)
    yaw = wrap_pi(yaw + yaw_rate * dt)
    px += v * math.sin(yaw) * dt
    py += v * math.cos(yaw) * dt
    return px, py, yaw, yaw_rate

# ===================== 10. NOMINAL PATH FOLLOWING =====================
MAX_STEER = 0.55
BASE_SPEED = 8.0
MIN_SPEED = 2.0

def nominal_path_steer(px, py, yaw, s_est):
    Ld = 6.0 + 0.3 * BASE_SPEED
    gx, gy = get_oval_pos(s_est + Ld, 0.0)
    goal_yaw = math.atan2((gx - px), (gy - py))
    yaw_err = wrap_pi(goal_yaw - yaw)
    return float(np.clip(1.2 * yaw_err, -MAX_STEER, +MAX_STEER))

# ===================== 11. ACTIVE AVOIDANCE =====================
STEER_ENVELOPE = 0.45
N_CAND = 9
FORWARD_HORIZON = 10.0
EVAL_STEPS = 10

W_CLEAR = 1.0
W_DEV = 0.35

OBSTACLE_THRESH = 8.0
CRITICAL_THRESH = 4.5

def evaluate_candidate(px, py, yaw, v, delta_cand, delta_nom, s_est_guess):
    dt_sim = FORWARD_HORIZON / EVAL_STEPS
    sim_px, sim_py, sim_yaw = px, py, yaw
    min_clear = 1e9
    cte_acc = 0.0
    s_est = s_est_guess

    for _ in range(EVAL_STEPS):
        sim_px, sim_py, sim_yaw, _ = integrate_bicycle(sim_px, sim_py, sim_yaw, v, delta_cand, dt_sim)
        min_clear = min(min_clear, clearance_at_point(sim_px, sim_py))
        s_est, _, _, _, _, _, _, cte = project_to_centerline(sim_px, sim_py, s_est, samples=11)
        cte_acc += abs(cte)

    avg_cte = cte_acc / EVAL_STEPS
    score = (W_CLEAR * min_clear) - (W_DEV * abs(delta_cand - delta_nom)) - (0.08 * avg_cte)
    return float(score), float(min_clear)

def active_avoidance(px, py, yaw, v, delta_nom, s_est_guess):
    cand = np.linspace(-STEER_ENVELOPE, +STEER_ENVELOPE, N_CAND)
    best_score = -1e9
    best_steer = float(delta_nom)
    best_clear = 1e9
    for d in cand:
        score, min_clear = evaluate_candidate(px, py, yaw, v, float(d), delta_nom, s_est_guess)
        if score > best_score:
            best_score = score
            best_steer = float(d)
            best_clear = float(min_clear)
    return best_steer, best_clear

# ===================== 12. ANTI-SPIN CONTROL =====================
STEER_RATE_LIMIT = 2.0
STEER_DAMP = 0.75
MAX_LAT_ACCEL = 5.0
MIN_SPEED_FOR_STEER = 0.5

def anti_spin(delta_cmd, delta_prev, v, dt):
    max_step = STEER_RATE_LIMIT * dt
    delta_rl = float(np.clip(delta_cmd, delta_prev - max_step, delta_prev + max_step))

    v_eff = max(v, MIN_SPEED_FOR_STEER)
    tan_lim = (MAX_LAT_ACCEL * WHEELBASE) / (v_eff * v_eff)
    delta_dyn = math.atan(max(0.0, tan_lim))
    delta_dyn = min(delta_dyn, MAX_STEER)

    delta_mag = float(np.clip(delta_rl, -delta_dyn, +delta_dyn))
    delta_out = STEER_DAMP * delta_prev + (1.0 - STEER_DAMP) * delta_mag
    return float(delta_out), float(delta_dyn)

# ===================== 13. ROBUST CAMERA RGB EXTRACTION =====================
def _looks_like_img(x):
    return isinstance(x, np.ndarray) and x.ndim == 3 and x.shape[-1] in (3, 4)

def _pick_img(ret):
    if ret is None:
        return None
    if _looks_like_img(ret):
        return ret
    if isinstance(ret, dict):
        for k in ("rgb", "rgba", "color", "image"):
            if k in ret and _looks_like_img(ret[k]):
                return ret[k]
        for v in ret.values():
            if _looks_like_img(v):
                return v
        return None
    if isinstance(ret, (tuple, list)):
        for it in ret:
            if _looks_like_img(it):
                return it
        for it in ret:
            img = _pick_img(it)
            if img is not None:
                return img
        return None
    return None

def render_rgb(cam):
    for fn in ("render", "render_rgb", "get_rgb", "rgb", "get_rgba"):
        f = getattr(cam, fn, None)
        if f is None:
            continue
        try:
            ret = f(rgb=True) if fn == "render" else f()
        except TypeError:
            try:
                ret = f()
            except Exception:
                continue
        except Exception:
            continue

        img = _pick_img(ret)
        if img is None:
            continue

        if img.dtype != np.uint8:
            mx = float(np.max(img)) if img.size else 1.0
            if mx <= 1.5:
                img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
            else:
                img = np.clip(img, 0, 255).astype(np.uint8)

        if img.shape[-1] == 4:
            img = img[..., :3]
        return img

    raise RuntimeError("Camera did not return an RGB frame.")

# ===================== 14. RUN + VIDEO =====================
out_dir = os.path.join(os.getcwd(), "videos-1")
os.makedirs(out_dir, exist_ok=True)
video_path_top = os.path.join(out_dir, "top_view.mp4")
video_path_incar = os.path.join(out_dir, "incar_view.mp4")

writer_top = imageio.get_writer(video_path_top, fps=video_fps)
writer_incar = imageio.get_writer(video_path_incar, fps=video_fps)

steps = int(25.0 / sim_dt)
px, py, pz, yaw, s_est = float(start_x), float(start_y - 5.0), 0.8, 0.0, 0.0
delta = 0.0

print("Starting 25s ACTIVE AVOIDANCE + ANTI-SPIN run with TOP-DOWN VIEW...")

try:
    for step in range(steps):
        # keep cones/obstacles fixed (pose pin)
        for ent, p, q in pinned_pose:
            ent.set_pos((float(p[0]), float(p[1]), float(p[2])))
            ent.set_quat((float(q[0]), float(q[1]), float(q[2]), float(q[3])))

        # projection
        s_est, _, _, _, _, _, _, cte = project_to_centerline(px, py, s_est, samples=15)

        # lidar trigger
        ranges = lidar_scan_fast(px, py, yaw, ray_angles, obs_boxes, LIDAR_MAX)
        front_min = float(np.min(ranges[(np.abs(ray_angles) < math.radians(20.0))]))

        # nominal steering
        delta_nom = nominal_path_steer(px, py, yaw, s_est)

        # active avoidance when needed
        if front_min < OBSTACLE_THRESH:
            delta_best, min_clear_best = active_avoidance(px, py, yaw, BASE_SPEED, delta_nom, s_est)
            w = float(np.clip((OBSTACLE_THRESH - front_min) / max(OBSTACLE_THRESH - CRITICAL_THRESH, 1e-6), 0.0, 1.0))
            delta_cmd = (1.0 - w) * delta_nom + w * delta_best
        else:
            delta_cmd = delta_nom
            min_clear_best = 1e9

        # speed rule
        speed = BASE_SPEED
        if front_min < CRITICAL_THRESH or min_clear_best < 1.8:
            speed = MIN_SPEED

        # anti-spin
        delta, delta_dyn_cap = anti_spin(delta_cmd, delta, speed, sim_dt)
        delta = float(np.clip(delta, -MAX_STEER, +MAX_STEER))

        # integrate kinematics
        px, py, yaw, _ = integrate_bicycle(px, py, yaw, speed, delta, sim_dt)

        # lane clamp
        s_est, cx, cy, _, _, nx, ny, cte = project_to_centerline(px, py, s_est, window=18.0, samples=21)
        if abs(cte) > LANE_SAFE:
            cte_clip = float(np.clip(cte, -LANE_SAFE, +LANE_SAFE))
            px = float(cx + cte_clip * nx)
            py = float(cy + cte_clip * ny)
            delta *= 0.6

        # set car pose (upright: pure yaw) - SINGLE CALL to yaw_to_quat_wxyz
        set_pose_yaw(car, px, py, pz, yaw)
        set_joint_positions([delta, delta])

        # step to sync transforms
        scene.step()

        # render/write video
        if step % steps_per_frame == 0:
            # Top camera is static - no updates needed
            frame_top = render_rgb(cam_top)
            writer_top.append_data(frame_top)
            
            # Update in-car camera to follow car
            update_incar_camera(cam_incar, px, py, pz, yaw)
            frame_incar = render_rgb(cam_incar)
            writer_incar.append_data(frame_incar)

        # log ~1 Hz
        if step % int(1.0 / sim_dt) == 0:
            t = step * sim_dt
            print(
                f"t={t:4.1f}s | v={speed:4.1f} | delta={delta:+.2f} (cap {delta_dyn_cap:+.2f}) | "
                f"cte={cte:+.2f} | front={front_min:4.1f}",
                flush=True
            )

finally:
    writer_top.close()
    writer_incar.close()
    print("\nSaved top view:", video_path_top)
    print("Saved in-car view:", video_path_incar)


[I 03/09/26 18:47:04.694 3904] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


LANE_SAFE = 3.2
Starting 25s ACTIVE AVOIDANCE + ANTI-SPIN run with TOP-DOWN VIEW...
t= 0.0s | v= 8.0 | delta=+0.00 (cap +0.22) | cte=-0.73 | front=25.2
t= 1.0s | v= 8.0 | delta=+0.00 (cap +0.22) | cte=+0.00 | front=17.2
t= 2.0s | v= 8.0 | delta=+0.00 (cap +0.22) | cte=-0.00 | front= 9.3
t= 3.0s | v= 2.0 | delta=+0.17 (cap +0.55) | cte=-0.03 | front= 6.3
t= 4.0s | v= 8.0 | delta=+0.01 (cap +0.22) | cte=-0.97 | front=19.3
t= 5.0s | v= 8.0 | delta=-0.13 (cap +0.22) | cte=-1.60 | front=11.4
t= 6.0s | v= 2.0 | delta=+0.08 (cap +0.55) | cte=-0.71 | front= 6.7
t= 7.0s | v= 2.0 | delta=-0.08 (cap +0.55) | cte=-0.37 | front= 4.4
t= 8.0s | v= 8.0 | delta=+0.07 (cap +0.22) | cte=+0.73 | front=30.0
t= 9.0s | v= 8.0 | delta=-0.20 (cap +0.22) | cte=+0.95 | front=30.0
t=10.0s | v= 8.0 | delta=-0.09 (cap +0.22) | cte=+0.83 | front=30.0
t=11.0s | v= 8.0 | delta=-0.22 (cap +0.22) | cte=+0.08 | front=30.0
t=12.0s | v= 8.0 | delta=-0.22 (cap +0.22) | cte=-0.06 | front=15.2
t=13.0s | v= 2.0 | delta=-0.11 (